# day-24-streaming — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [8]:
# ---- Solution 4 ----
def two_tool_stream():
    yield {"type": "message_start", "message": {}}
    yield {"type": "content_block_start", "index": 0, "content_block": {"type": "tool_use", "id": "t1", "name": "get_weather", "input": {}}}
    yield {"type": "content_block_start", "index": 1, "content_block": {"type": "tool_use", "id": "t2", "name": "convert", "input": {}}}
    a, b = json.dumps({"city": "Paris"}), json.dumps({"amount": 50})
    for i in range(0, max(len(a), len(b)), 4):
        if i < len(a): yield {"type": "content_block_delta", "index": 0, "delta": {"type": "input_json_delta", "partial_json": a[i:i+4]}}
        if i < len(b): yield {"type": "content_block_delta", "index": 1, "delta": {"type": "input_json_delta", "partial_json": b[i:i+4]}}
    yield {"type": "content_block_stop", "index": 0}
    yield {"type": "content_block_stop", "index": 1}
    yield {"type": "message_delta", "delta": {"stop_reason": "tool_use"}}
    yield {"type": "message_stop"}

bufs = {}
for ev in two_tool_stream():
    if ev["type"] == "content_block_start" and ev["content_block"]["type"] == "tool_use":
        bufs[ev["index"]] = {"name": ev["content_block"]["name"], "json": ""}
    elif ev["type"] == "content_block_delta" and ev["delta"]["type"] == "input_json_delta":
        bufs[ev["index"]]["json"] += ev["delta"]["partial_json"]
    elif ev["type"] == "content_block_stop" and ev["index"] in bufs:
        bufs[ev["index"]]["args"] = json.loads(bufs[ev["index"]]["json"])
print("S4:", [(b["name"], b["args"]) for b in bufs.values()])

S4: [('get_weather', {'city': 'Paris'}), ('convert', {'amount': 50})]


In [9]:
# ---- Solution 5 ----
frag = ""
errors_until_end = 0
blob = json.dumps({"city": "Paris", "units": "celsius"})
for i in range(0, len(blob), 5):
    frag += blob[i:i+5]
    try:
        json.loads(frag)
    except json.JSONDecodeError:
        errors_until_end += 1
print(f"S5: {errors_until_end} JSONDecodeErrors while fragments were partial; "
      f"parse succeeds only once frag == the complete string.")

S5: 7 JSONDecodeErrors while fragments were partial; parse succeeds only once frag == the complete string.


### Solutions 1, 2, 3, 6 (sketch)

**S1:** keep a `word` buffer; on each `text_delta`, split on spaces — append complete words to
a `line`, flush `line` + newline when `len(line) + len(word) > 72`, keep the trailing partial
word in `word`.

**S2:** `first_token_t - start_t` is your TTFT; `stop_t - start_t` is total. Smaller `chunk`
(more, smaller deltas) barely changes TTFT but adds per-event overhead; TTFT is dominated by
the model's prefill, not by chunking.

**S3:** emit `content_block_start {type:"thinking"}` → `thinking_delta`s →
`content_block_stop`, then the text block at index 1. `reduce_stream` already handles
`thinking_delta`; the final `content` has `[{type:"thinking",...}, {type:"text",...}]`. Render
thinking with a dim style and don't send it to the user as the answer.

**S6:** break out of the `for piece in text_stream` loop early. The HTTP connection closes,
but the server had already generated some or all of the response — **you're billed for the
output tokens the server produced**, not for what you read. `get_final_message()` won't have
the full text if you stopped early; check `usage` from the last `message_delta` you saw.

### Answer key
1. Any two: lower time-to-first-token / better perceived latency; avoiding HTTP timeouts on
   large `max_tokens` (streaming is required there); incremental UI (typewriter, live tool
   display); acting on partial output during long agent turns.
2. `message_start` → `content_block_start` (text) → `content_block_delta` (text_delta) × N →
   `content_block_stop` → `message_delta` (stop_reason, usage) → `message_stop`.
3. It's a fragment of a JSON string, not valid JSON on its own. You concatenate all
   `partial_json` pieces for that block and parse once at `content_block_stop`.
4. `text_stream` yields only the text pieces, already filtered and accumulated for you — the
   simple path. Raw events give you tool-call JSON deltas, thinking deltas, block boundaries,
   and usage — needed for live tool-call UIs or thinking display.
5. Catch the exception around the loop; show the accumulated partial with an "(interrupted)"
   marker or discard it; retry as a **fresh** request. You cannot resume a stream.
6. Batch/offline jobs; when you need the complete output before acting on it (JSON parsing,
   routing); tiny classification outputs; inside a tool loop where you only want the final
   message.
7. The output tokens the server already generated before the connection closed — not just
   what you received. Streaming doesn't reduce token cost, only latency.